In [1]:
import heapq
from bisect import bisect_left, bisect_right
from collections import Counter, defaultdict, deque
from functools import cache

# LeetCode Playbook

---
# 1) Complexity Budget

LeetCode allows roughly $10^8$ simple operations per second in compiled languages and about
$10^7$ in Python. The input size therefore tells you which complexity the intended solution
has — often the strongest hint in the whole statement.

**Cost of common Python operations**

| Operation | Cost | Note |
|---|---|---|
| `list.append`, `list.pop()` | O(1) | end of list |
| `list.pop(0)`, `list.insert(0, x)`, `x in list` | O(n) | use `deque` / `set` instead |
| `deque.appendleft`, `deque.popleft` | O(1) | queues and BFS |
| `dict` / `set` lookup, insert, delete | O(1) average | keys must be hashable (tuples, not lists) |
| `heapq.heappush`, `heappop` | O(log n) | min-heap; negate for max-heap |
| `heapq.heapify` | O(n) | faster than n pushes |
| `bisect_left` / `bisect_right` | O(log n) | on a sorted list |
| `sorted`, `list.sort` | O(n log n) | stable; `key=` for custom order |
| `s[i:j]`, `s + t` | O(length) | slicing copies |
| `"".join(parts)` | O(total) | build strings with a list, not `+=` in a loop |

---
# 2) Python Toolkit

The standard library replaces most hand-written data structures. These are the calls that
appear across the practice notebooks.

In [2]:
# Counter: frequencies, multiset arithmetic, top-k
count = Counter("mississippi")
print(count.most_common(2), Counter("aab") - Counter("ab"))

# defaultdict: grouping without key checks
groups = defaultdict(list)
for word in ["eat", "tea", "tan"]:
    groups["".join(sorted(word))].append(word)
print(dict(groups))

# deque: O(1) at both ends for queues and sliding windows
dq = deque([1, 2, 3])
dq.appendleft(0)
dq.pop()
print(dq)

[('i', 4), ('s', 4)] Counter({'a': 1})
{'aet': ['eat', 'tea'], 'ant': ['tan']}
deque([0, 1, 2])


In [3]:
# heapq: min-heap; negate values for a max-heap; tuples compare element-wise
heap = [5, 1, 8, 3]
heapq.heapify(heap)
print(heapq.heappop(heap), heapq.nsmallest(2, heap), heapq.nlargest(1, heap))
max_heap = [-x for x in [5, 1, 8]]
heapq.heapify(max_heap)
print(-max_heap[0])

# bisect: insertion points in a sorted list
sorted_nums = [1, 3, 3, 3, 7]
print(bisect_left(sorted_nums, 3), bisect_right(sorted_nums, 3))


# cache: turns a recursive function into top-down DP
@cache
def ways(n: int) -> int:
    """Stair-climbing ways with 1- or 2-steps."""
    return 1 if n <= 1 else ways(n - 1) + ways(n - 2)


print(ways(50))

1 [3, 5] [8]
8
1 4
20365011074


In [4]:
# sorting with keys: by length, then alphabetically; descending numbers
words = ["bb", "a", "ccc", "ab"]
print(sorted(words, key=lambda w: (len(w), w)))
print(sorted([3, 1, 2], reverse=True))

# grid neighbours and bounds checks
DIRS = [(1, 0), (-1, 0), (0, 1), (0, -1)]
rows, cols, r, c = 3, 3, 0, 0
print(
    [(r + dr, c + dc) for dr, dc in DIRS if 0 <= r + dr < rows and 0 <= c + dc < cols]
)

# infinities, integer division and modulo with negatives
print(float("inf") > 10**18, -7 // 2, int(-7 / 2), -7 % 3)

['a', 'ab', 'bb', 'ccc']
[3, 2, 1]
[(1, 0), (0, 1)]
True -4 -3 2


**Python gotchas that cause wrong answers**

| Gotcha | Fix |
|---|---|
| `[[0] * n] * m` shares one row m times | `[[0] * n for _ in range(m)]` |
| `-7 // 2 == -4` (floor), problems often want truncation | `int(a / b)` |
| default recursion limit 1000 | iterative DFS / BFS, or `sys.setrecursionlimit` |
| mutating a list while iterating | iterate over a copy or build a new list |
| `path` appended by reference in backtracking | `out.append(path.copy())` |
| mutable default argument `def f(x=[])` | `def f(x=None)` |
| `heapq` compares the second tuple element on ties | add a counter/index as a tie-breaker |

---
# 3) Pattern Recognition

Read the statement for these cues. The left column is what the problem *says*; the right
column is what to try first.

| If the problem says… | Try | Practice |
|---|---|---|
| "pair / triple with sum", "seen before", "group by" | hash map / set | 02 §1 |
| sorted array, "pair", "in place", palindrome | two pointers | 02 §2 |
| "longest / shortest substring or subarray with …" | sliding window | 02 §3 |
| "subarray sum equals / divisible by k", negatives allowed | prefix sum + hash map | 02 §4 |
| brackets, nested structure, "previous / next greater" | stack / monotonic stack | 03 §1 |
| sorted or rotated input, "minimum X such that feasible" | binary search (on the answer) | 03 §2 |
| linked list: middle, cycle, k-th from end, reverse | slow/fast pointers, dummy head | 03 §3 |
| tree: depth, path, ancestor, balanced | DFS returning values upward | 04 §1 |
| tree: level, right view, nearest | BFS by level | 04 §2 |
| BST, "k-th smallest", sorted order | in-order traversal / bounds | 04 §3 |
| prefixes, autocomplete, many words in a grid | trie | 04 §4 |
| "top k", "k-th largest", streaming median, merge k | heap | 04 §5 |
| grid regions, islands, flood fill | DFS / BFS on grid | 05 §1 |
| "minimum steps", unweighted shortest path | BFS (multi-source if many starts) | 05 §1–2 |
| prerequisites, dependency order | topological sort | 05 §3 |
| "connected?", merging groups, redundant edge | union-find | 05 §4 |
| weighted shortest path, "cheapest", "minimum effort" | Dijkstra / Bellman–Ford | 05 §5 |
| "all combinations / permutations / subsets", n ≤ 20 | backtracking | 05 §6 |
| "number of ways", "min/max cost", overlapping choices | dynamic programming | 06 |
| two strings compared character by character | 2D DP over prefixes | 06 §5 |
| intervals: merge, overlap, rooms | sort by start / end, heap of ends | 07 §2 |
| "minimum number of …" with a clear local choice | greedy (prove it!) | 07 §1 |
| "without extra space", appears twice/once | XOR / bit tricks | 07 §4 |
| "design a class with O(1) …" | hash map + list / deque / heap combination | 07 §5 |

---
# 4) Pattern Templates

Each template is the skeleton that the practice questions specialise. Type each one from
memory until you can do it without hesitation — this is the fastest way to remove
off-by-one and boundary bugs.

## 4.1 Searching templates

In [5]:
def lower_bound(nums: list[int], target: int) -> int:
    """First index i with nums[i] >= target (len(nums) if none)."""
    lo, hi = 0, len(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] >= target:
            hi = mid
        else:
            lo = mid + 1
    return lo


def min_feasible(lo: int, hi: int, feasible) -> int:
    """Smallest value in [lo, hi] where the monotone predicate is True."""
    while lo < hi:
        mid = (lo + hi) // 2
        if feasible(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo


print(lower_bound([1, 3, 5, 7], 4), min_feasible(1, 100, lambda x: x * x >= 50))

2 8


## 4.2 Array scan templates

In [6]:
def longest_valid_window(nums: list[int], limit: int) -> int:
    """Longest subarray whose sum stays <= limit (non-negative values)."""
    left = total = best = 0
    for right, x in enumerate(nums):
        # extend the window
        total += x
        # shrink until valid again
        while total > limit:
            total -= nums[left]
            left += 1
        best = max(best, right - left + 1)
    return best


def count_subarrays_with_sum(nums: list[int], k: int) -> int:
    """Subarrays summing to k (negatives allowed) via prefix sums."""
    seen, prefix, total = Counter({0: 1}), 0, 0
    for x in nums:
        prefix += x
        total += seen[prefix - k]
        seen[prefix] += 1
    return total


def next_greater(nums: list[int]) -> list[int]:
    """Next greater element to the right (-1 if none), monotonic stack."""
    out, stack = [-1] * len(nums), []
    for i, x in enumerate(nums):
        while stack and nums[stack[-1]] < x:
            out[stack.pop()] = x
        stack.append(i)
    return out


print(longest_valid_window([2, 1, 3, 1, 1], 5))
print(count_subarrays_with_sum([1, -1, 1, 1], 1), next_greater([2, 1, 3]))

3
5 [3, 3, -1]


## 4.3 Graph templates

In [7]:
def bfs_distance(graph: dict[int, list[int]], start: int) -> dict[int, int]:
    """Unweighted shortest distances from start."""
    dist = {start: 0}
    queue = deque([start])
    while queue:
        u = queue.popleft()
        for v in graph[u]:
            if v not in dist:
                dist[v] = dist[u] + 1
                queue.append(v)
    return dist


def topological_order(n: int, edges: list[tuple[int, int]]) -> list[int]:
    """Kahn's algorithm; shorter than n means a cycle exists."""
    graph, indeg = defaultdict(list), [0] * n
    for u, v in edges:
        graph[u].append(v)
        indeg[v] += 1
    queue = deque(i for i in range(n) if indeg[i] == 0)
    order = []
    while queue:
        u = queue.popleft()
        order.append(u)
        for v in graph[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                queue.append(v)
    return order


def dijkstra(graph: dict[int, list[tuple[int, int]]], src: int) -> dict[int, int]:
    """Shortest distances with non-negative weights."""
    dist, heap = {src: 0}, [(0, src)]
    while heap:
        d, u = heapq.heappop(heap)
        # skip stale heap entries
        if d > dist[u]:
            continue
        for v, w in graph[u]:
            if d + w < dist.get(v, float("inf")):
                dist[v] = d + w
                heapq.heappush(heap, (d + w, v))
    return dist


g = defaultdict(list, {0: [1, 2], 1: [3], 2: [3]})
print(bfs_distance(g, 0), topological_order(4, [(0, 1), (0, 2), (1, 3), (2, 3)]))
print(dijkstra(defaultdict(list, {0: [(1, 4), (2, 1)], 2: [(1, 2)]}), 0))

{0: 0, 1: 1, 2: 1, 3: 2} [0, 1, 2, 3]
{0: 0, 1: 3, 2: 1}


In [8]:
class UnionFind:
    """Disjoint sets with path compression and union by size."""

    def __init__(self, n: int) -> None:
        self.parent = list(range(n))
        self.size = [1] * n

    def find(self, x: int) -> int:
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a: int, b: int) -> bool:
        """Join the sets of a and b; False if already joined."""
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.size[ra] < self.size[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        self.size[ra] += self.size[rb]
        return True


uf = UnionFind(4)
print(uf.union(0, 1), uf.union(1, 0), uf.find(1) == uf.find(0))

True False True


## 4.4 Backtracking and DP templates

In [9]:
def all_subsets(nums: list[int]) -> list[list[int]]:
    """Choose, explore, un-choose."""
    out, path = [], []

    def backtrack(start: int) -> None:
        out.append(path.copy())
        for i in range(start, len(nums)):
            path.append(nums[i])
            backtrack(i + 1)
            path.pop()

    backtrack(0)
    return out


def knapsack_01(weights: list[int], values: list[int], capacity: int) -> int:
    """Best value with each item used at most once."""
    dp = [0] * (capacity + 1)
    for w, v in zip(weights, values):
        # iterate capacity downwards so each item is used once
        for c in range(capacity, w - 1, -1):
            dp[c] = max(dp[c], dp[c - w] + v)
    return dp[capacity]


def lcs(a: str, b: str) -> int:
    """Two-string DP over prefixes."""
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[-1][-1]


print(all_subsets([1, 2]), knapsack_01([1, 3, 4], [15, 20, 30], 4), lcs("abcde", "ace"))

[[], [1], [1, 2], [2]] 35 3
